← [15 - Tableaux imbriqués et grilles 2D](Z3-15-Nested-Arrays-2D-Python.ipynb) | [README Z3](README.md) | [16b - couche de données >>](Z3-16b-Meal-Planner-Data-External-Python.ipynb)

# 16. Meal-Planner déclaratif : du modèle Z3 au plan hebdomadaire (C# / Microsoft.Z3 natif)

Après les puzzles (Sudoku, carré magique) et les grilles 2D, on applique la programmation
par contraintes à un **problème d'optimisation réaliste** : composer un **menu équilibré** à
partir d'une base d'aliments, puis un **plan hebdomadaire** complet. C'est le saut de la
satisfiabilité (« existe-t-il une solution ? ») à l'**optimisation sous contraintes**
(« quelle est la solution la *moins chère* / la *plus variée* ? »).

Ce notebook modélise le problème en **C# avec Microsoft.Z3** (binding natif) et compare
**trois approches de modélisation** d'un même problème — index-based, énumération, booléenne —
puis montre comment `MkOptimize` trouve le menu minimal en coût, et comment un modèle à matrice
booléenne `jours × plats` produit un **plan hebdomadaire** que ne sait pas énumérer une
heuristique gloutonne (prong B : le couplage fenêtre-calorique × variété est non compositionnel).
Le jumeau Python (z3-py) de ce notebook est [Z3-16-Meal-Planner-Python](Z3-16-Meal-Planner-Python.ipynb).

> **Aller plus loin : la jambe corpus réel.** Ce notebook pose la modélisation sur un corpus
> **jouet** (10 puis 9 plats saisis à la main). La track se poursuit sur un
> **corpus réel** Ciqual × RecipeML (R ≈ 594 recettes) :
> [16b — couche de données](Z3-16b-Meal-Planner-Data-External-Python.ipynb) ·
> [16c — capstone patient](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) ·
> [16d — convergence à l'échelle](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) ·
> [16e — optimisation](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb).


In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System;
using System.Diagnostics;
using System.Linq;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs).
Z3NativeLoader.Register(typeof(Context).Assembly);

// Contexte Z3 partage par toutes les cellules (meme moteur C++ que le jumeau Python z3-solver).
var ctx = new Context();
Console.WriteLine($"Moteur : Microsoft.Z3 {Microsoft.Z3.Version.FullVersion}");

// Base de donnees nutritionnelle : (nom, kcal, proteines_g, cout_centimes, categorie).
(string Nom, int Kcal, int Prot, int Cout, string Cat)[] FOODS = {
    ("Salade verte",         80,  2,  300, "entree"),
    ("Soupe de legumes",    120,  4,  250, "entree"),
    ("Carpaccio",           150, 15,  600, "entree"),
    ("Poulet roti",         350, 30,  500, "plat"),
    ("Saumon grille",       300, 28,  800, "plat"),
    ("Pates bolognaise",    450, 18,  350, "plat"),
    ("Risotto",             400, 10,  400, "plat"),
    ("Salade de fruits",    100,  1,  200, "dessert"),
    ("Mousse au chocolat", 200,  5,  250, "dessert"),
    ("Tiramisu",            250,  7,  350, "dessert"),
};
var CATS = FOODS.Select(f => f.Cat).ToList();
string[] CATS_SET = { "entree", "plat", "dessert" };
int nEntree = CATS.Count(c => c == "entree");
int nPlat   = CATS.Count(c => c == "plat");
int nDessert= CATS.Count(c => c == "dessert");
Console.WriteLine($"Base nutritionnelle : {FOODS.Length} aliments ({nEntree} entrees, {nPlat} plats, {nDessert} desserts).");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Moteur : Microsoft.Z3 Z3 4.12.2.0


Base nutritionnelle : 10 aliments (3 entrees, 4 plats, 3 desserts).


In [2]:
// Contrainte 'exactement un vrai' parmi une liste de Bool z3.
// MkPBEq natif : somme ponderee (coeffs tous 1) egale 1 -- c'est-a-dire 'au moins un' ET
// 'au plus un' en UNE assertion, au lieu de l'encodage naif (Or + exclusion pairwise en O(n^2)).
// Idiom MkPBEq retrouve a l'echelle reelle dans les notebooks 16c/16d (cf. forward-links en conclusion).
void ExactementUn(Solver s, BoolExpr[] bs) =>
    s.Assert(ctx.MkPBEq(Enumerable.Repeat(1, bs.Length).ToArray(), bs, 1));
// Surcharge pour Optimize (S4) : Solver et Optimize n'ont pas de type base commun pour Assert.
void ExactementUn(Optimize o, BoolExpr[] bs) =>
    o.Assert(ctx.MkPBEq(Enumerable.Repeat(1, bs.Length).ToArray(), bs, 1));

Console.WriteLine("Helper ExactementUn defini (MkPBEq natif : exactement-un en 1 assertion, vs O(n^2) pairwise).");


Helper ExactementUn defini (MkPBEq natif : exactement-un en 1 assertion, vs O(n^2) pairwise).


## 1. Approche index-based : un `Int` par catégorie

L'encodage le plus direct : une variable `Int` par catégorie (`ctx.MkIntConst`), dont la
valeur est l'**index** de l'aliment choisi. La contrainte de bornes (`0 <= idx <= taille-1`)
suffit à garantir un choix valide. Sans contrainte nutritionnelle, le solveur renvoie la
première affectation (souvent les index 0). Cette approche est simple mais **mal adaptée aux
contraintes nutritionnelles** (l'index n'a pas de relation linéaire naturelle avec les
kcal/protéines — il faut une table de lookup, moins élégante que l'approche booléenne suivante).


In [3]:
// (1) Approche index-based : un Int par categorie ; resolution des index vers les aliments
//     par lookup C# HORS solveur (l'index est un handle opaque).
Solver Si = ctx.MkSolver();
IntExpr ie  = ctx.MkIntConst("idx_entree");
IntExpr ip  = ctx.MkIntConst("idx_plat");
IntExpr idc = ctx.MkIntConst("idx_dessert");
Si.Assert(ctx.MkAnd(
    ctx.MkGe(ie, ctx.MkInt(0)),  ctx.MkLe(ie, ctx.MkInt(2)),
    ctx.MkGe(ip, ctx.MkInt(0)),  ctx.MkLe(ip, ctx.MkInt(3)),
    ctx.MkGe(idc, ctx.MkInt(0)), ctx.MkLe(idc, ctx.MkInt(2))));
if (Si.Check() != Status.SATISFIABLE) throw new InvalidOperationException("S1 : UNSAT inattendu");
var mi = Si.Model;
int[] IENT  = Enumerable.Range(0, FOODS.Length).Where(i => CATS[i] == "entree").ToArray();
int[] IPLAT = Enumerable.Range(0, FOODS.Length).Where(i => CATS[i] == "plat").ToArray();
int[] IDES  = Enumerable.Range(0, FOODS.Length).Where(i => CATS[i] == "dessert").ToArray();
int e = IENT[(int)((IntNum)mi.Evaluate(ie,  true)).Int64];
int p = IPLAT[(int)((IntNum)mi.Evaluate(ip, true)).Int64];
int d = IDES[(int)((IntNum)mi.Evaluate(idc, true)).Int64];
int cal = FOODS[e].Kcal + FOODS[p].Kcal + FOODS[d].Kcal;
Console.WriteLine("Menu index-based (sans contrainte nutritionnelle) :");
Console.WriteLine($"  Entree  : {FOODS[e].Nom}");
Console.WriteLine($"  Plat    : {FOODS[p].Nom}");
Console.WriteLine($"  Dessert : {FOODS[d].Nom}");
Console.WriteLine($"  Total   : {cal} kcal");


Menu index-based (sans contrainte nutritionnelle) :


  Entree  : Salade verte


  Plat    : Poulet roti


  Dessert : Salade de fruits


  Total   : 530 kcal


### Interprétation 1 — index vs caractéristiques

L'index est un handle opaque : le solveur ne « sait » pas que l'index 0 (Salade) a 80 kcal et
l'index 2 (Carpaccio) en a 150. Pour ajouter une contrainte « total entre 600 et 900 kcal »,
il faut relier l'index aux caractéristiques, ce qui demande une table de lookup ou un grand
`if`/`else` en C# — verbeux. L'approche booléenne (§3) rend cette relation **native**.


## 2. Approche par énumération + filtrage

Avant Z3, on résoudrait ce problème par **énumération exhaustive** : générer tous les triples
(entree, plat, dessert), filtrer ceux qui satisfont les contraintes nutritionnelles, puis
prendre le moins cher. Le C# n'a pas d'`itertools.product` — trois boucles imbriquées font le
produit cartésien. Cela marche pour 3×4×3 = 36 combinaisons, mais **explose** avec la taille
(un plan hebdomadaire à 9 plats³/jour × 5 jours = ~59000 milliards de combinaisons). Cette
approche sert de **baseline de complexité** pour mesurer ce que Z3 évite.


In [4]:
// (2) Approche par enumeration + filtrage : trois boucles imbriquees font le produit carésien
//     (le C# n'a pas d'itertools.product). Baseline de complexite -- explose a l'echelle.
var sw1 = Stopwatch.StartNew();
var valides = new List<(int e, int p, int d, int cal, int prot, int cost)>();
foreach (var e in IENT)
foreach (var p in IPLAT)
foreach (var d in IDES) {
    int cal  = FOODS[e].Kcal + FOODS[p].Kcal + FOODS[d].Kcal;
    int prot = FOODS[e].Prot + FOODS[p].Prot + FOODS[d].Prot;
    int cost = FOODS[e].Cout + FOODS[p].Cout + FOODS[d].Cout;
    if (cal >= 600 && cal <= 900 && prot >= 25 && cost <= 1200)
        valides.Add((e, p, d, cal, prot, cost));
}
sw1.Stop();
int ncomb = IENT.Length * IPLAT.Length * IDES.Length;
Console.WriteLine($"Enumeration : {valides.Count} menus valides sur {ncomb} combinaisons en {sw1.Elapsed.TotalMilliseconds:F2} ms");
if (valides.Count > 0) {
    var best = valides.MinBy(v => v.cost);
    Console.WriteLine($"Menu le moins cher : {FOODS[best.e].Nom} + {FOODS[best.p].Nom} + {FOODS[best.d].Nom}");
    Console.WriteLine($"  -> {best.cal} kcal, {best.prot}g prot, {best.cost / 100.0:F2} EUR");
}


Enumeration : 11 menus valides sur 36 combinaisons en 0,41 ms


Menu le moins cher : Soupe de legumes + Pates bolognaise + Mousse au chocolat


  -> 770 kcal, 27g prot, 8,50 EUR


### Interprétation 2 — la malédiction de la dimension

36 combinaisons, c'est instantané. Mais chaque dimension supplémentaire multiplie le coût :
passer à 4 catégories de 10 aliments = 10⁴ = 10000 ; un plan sur 5 jours avec 3 choix/jour =
~1.7×10⁷. L'énumération est en O(produit des domaines) — **exponentiel en le nombre de
variables**. Z3, lui, raisonne sur les contraintes **symboliquement** (propagation +
apprentissage de clauses), sans énumérer — c'est tout l'intérêt de l'approche booléenne
suivante.


## 3. Approche booléenne : une variable `Bool` par aliment

L'encodage idiomatique : **une variable booléenne par aliment** (`sel_i` = « l'aliment i est
choisi ») via `ctx.MkBoolConst`. Les contraintes deviennent naturelles :

- **Exactement 1 entrée/plat/dessert** : `MkPBEq` natif (helper `ExactementUn`).
- **Total calorique** : `MkAdd` de `MkITE(sel_i, kcal_i, 0)` — la table de lookup disparaît,
  chaque aliment **porte** ses caractéristiques dans l'expression.

Le solveur explore l'espace des affectations booléennes par propagation, sans énumérer.


In [5]:
// (3) Approche booleenne : une variable Bool par aliment ; MkITE(sel,val,0) comme table de
//     lookup symbolique (chaque aliment PORTE ses caracteristiques dans l'expression).
Solver Sb = ctx.MkSolver();
BoolExpr[] sel = Enumerable.Range(0, FOODS.Length).Select(i => ctx.MkBoolConst($"sel_{i}")).ToArray();
foreach (var cat in CATS_SET)
    ExactementUn(Sb, Enumerable.Range(0, sel.Length).Where(idx => CATS[idx] == cat).Select(idx => sel[idx]).ToArray());
IntExpr totalCal  = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(sel[i], ctx.MkInt(FOODS[i].Kcal), ctx.MkInt(0))).ToArray());
IntExpr totalProt = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(sel[i], ctx.MkInt(FOODS[i].Prot), ctx.MkInt(0))).ToArray());
IntExpr totalCost = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(sel[i], ctx.MkInt(FOODS[i].Cout), ctx.MkInt(0))).ToArray());
Sb.Assert(ctx.MkAnd(ctx.MkGe(totalCal, ctx.MkInt(600)), ctx.MkLe(totalCal, ctx.MkInt(900))));
Sb.Assert(ctx.MkGe(totalProt, ctx.MkInt(25)));
Sb.Assert(ctx.MkLe(totalCost, ctx.MkInt(1200)));
if (Sb.Check() == Status.SATISFIABLE) {
    var mb = Sb.Model;
    var choisi = Enumerable.Range(0, FOODS.Length)
        .Where(i => ((BoolExpr)mb.Evaluate(sel[i], true)).BoolValue == Z3_lbool.Z3_L_TRUE)
        .Select(i => (FOODS[i].Nom, CATS[i], FOODS[i].Kcal, FOODS[i].Prot, FOODS[i].Cout)).ToList();
    Console.WriteLine("Menu equilibre (approche booleenne) :");
    int cCal = 0, cProt = 0, cCost = 0;
    foreach (var (nom, cat, k, pr, co) in choisi) {
        Console.WriteLine($"  [{cat,-8}] {nom,-20} {k,4} kcal, {pr,2}g prot, {co / 100.0:F2} EUR");
        cCal += k; cProt += pr; cCost += co;
    }
    Console.WriteLine($"  --- Total : {cCal} kcal, {cProt}g proteines, {cCost / 100.0:F2} EUR ---");
} else {
    Console.WriteLine("UNSAT : pas de menu equilibre dans ces bornes.");
}


Menu equilibre (approche booleenne) :


  [entree  ] Soupe de legumes      120 kcal,  4g prot, 2,50 EUR


  [plat    ] Poulet roti           350 kcal, 30g prot, 5,00 EUR


  [dessert ] Mousse au chocolat    200 kcal,  5g prot, 2,50 EUR


  --- Total : 670 kcal, 39g proteines, 10,00 EUR ---


### Interprétation 3 — `MkITE(bool, val, 0)` comme table de lookup symbolique

L'expression `MkITE(sel_i, kcal_i, 0)` est l'équivalent symbolique d'une table de lookup :
elle « active » la caractéristique de l'aliment i si et seulement s'il est sélectionné. Le
solveur propage ces sommes linéaires sans énumération — c'est ce qui rend l'approche booléenne
**passage à l'échelle** là où l'énumération explose.


## 4. Optimisation : le menu équilibré le moins cher

`MkSolver` répond sat/unsat ; `MkOptimize` ajoute un **objectif** (minimiser ou maximiser)
sous les mêmes contraintes. On demande ici le menu équilibré (600-900 kcal, ≥25g prot) de
**coût minimal**. `MkMinimize(cout)` pousse le solveur vers le coût le plus bas en un seul
appel — là où une recherche dichotomique manuelle nécessiterait N appels `MkSolver`.


In [6]:
// (4) Optimisation : le menu equilibre le moins cher. MkMinimize en UN appel (pas de dichotomie).
var sw2 = Stopwatch.StartNew();
Optimize OmC = ctx.MkOptimize();
BoolExpr[] selc = Enumerable.Range(0, FOODS.Length).Select(i => ctx.MkBoolConst($"sc_{i}")).ToArray();
foreach (var cat in CATS_SET)
    ExactementUn(OmC, Enumerable.Range(0, selc.Length).Where(idx => CATS[idx] == cat).Select(idx => selc[idx]).ToArray());
IntExpr tcCal  = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(selc[i], ctx.MkInt(FOODS[i].Kcal), ctx.MkInt(0))).ToArray());
IntExpr tcProt = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(selc[i], ctx.MkInt(FOODS[i].Prot), ctx.MkInt(0))).ToArray());
IntExpr tcCost = (IntExpr)ctx.MkAdd(Enumerable.Range(0, FOODS.Length).Select(i => (ArithExpr)ctx.MkITE(selc[i], ctx.MkInt(FOODS[i].Cout), ctx.MkInt(0))).ToArray());
OmC.Assert(ctx.MkAnd(ctx.MkGe(tcCal, ctx.MkInt(600)), ctx.MkLe(tcCal, ctx.MkInt(900))));
OmC.Assert(ctx.MkGe(tcProt, ctx.MkInt(25)));
OmC.MkMinimize(tcCost);
if (OmC.Check() != Status.SATISFIABLE) throw new InvalidOperationException("S4 : UNSAT inattendu");
sw2.Stop();
var moc = OmC.Model;
var choisiC = Enumerable.Range(0, FOODS.Length)
    .Where(i => ((BoolExpr)moc.Evaluate(selc[i], true)).BoolValue == Z3_lbool.Z3_L_TRUE)
    .Select(i => (FOODS[i].Nom, FOODS[i].Kcal, FOODS[i].Prot, FOODS[i].Cout)).ToList();
int minCost = choisiC.Sum(x => x.Item4);
Console.WriteLine($"Menu MIN-COUT (MkMinimize, {sw2.Elapsed.TotalMilliseconds:F1} ms) :");
foreach (var (nom, k, pr, co) in choisiC)
    Console.WriteLine($"  {nom,-20} {k,4} kcal, {pr,2}g prot, {co / 100.0:F2} EUR");
Console.WriteLine($"  --- Cout total minimal : {minCost / 100.0:F2} EUR ---");


Menu MIN-COUT (MkMinimize, 4,5 ms) :


  Soupe de legumes      120 kcal,  4g prot, 2,50 EUR


  Pates bolognaise      450 kcal, 18g prot, 3,50 EUR


  Mousse au chocolat    200 kcal,  5g prot, 2,50 EUR


  --- Cout total minimal : 8,50 EUR ---


### Interprétation 4 — `MkMinimize` vs dichotomie

`MkMinimize` encapsule la recherche d'optimum : le solveur combine la satisfiabilité des
contraintes avec la poursuite de l'objectif, en utilisant des algorithmes spécialisés (OBBTP,
maxres selon la théorie). L'alternative manuelle — une **dichotomie** sur le coût (`MkSolver`
avec `cost <= mid`, resserrer `mid` jusqu'à unsat) — marche mais demande N appels et une
logique de boucle. `MkOptimize` le fait en un appel, et généralise (`MkMaximize`, objectifs
multiples, front de Pareto — cf. notebook 16e).


## 5. Plan hebdomadaire : la matrice booléenne `jours × plats` (prong B)

Le morceau de résistance : composer un **plan sur D jours** où chaque jour a 1 petit-déjeuner,
1 déjeuner, 1 dîner, chaque plat servi **au plus une fois dans la semaine** (variété), et
chaque jour dans une **fenêtre calorique** `[DAY_LO, DAY_HI]`. L'encodage : une matrice
booléenne `Sel[j, i]` (jour j, plat i) — un tableau `BoolExpr[,]` en C#.

**Pourquoi c'est le prong B** : une heuristique **gloutonne** (remplir jour par jour avec le
premier trio valide restant) **se bloque** — la combinatoire « fenêtre × variété » n'est pas
compositionnelle (un choix local optimal rend le lendemain infaisable). Z3, lui, raisonne sur
**tous les jours simultanément** et trouve un plan global. C'est là que le solveur démontre sa
valeur par rapport à une heuristique naïve.


In [7]:
// Corpus de plats (S5) : (nom, categorie, kcal, proteines_g, cout_centimes).
(string Nom, string Cat, int Kcal, int Prot, int Cout)[] DISHES = {
    ("Porridge fruits rouges", "breakfast", 300, 12, 180),
    ("Yaourt granola",         "breakfast", 350, 18, 220),
    ("Oeufs brouilles",        "breakfast", 400, 22, 280),
    ("Salade cesar",           "lunch",     500, 25, 650),
    ("Wrap poulet",            "lunch",     550, 30, 580),
    ("Quiche epinards",        "lunch",     600, 20, 520),
    ("Poulet curry",           "dinner",    650, 35, 700),
    ("Saumon four",            "dinner",    700, 32, 850),
    ("Risotto champignons",    "dinner",    750, 18, 600),
};
string[] DC = DISHES.Select(d => d.Cat).ToArray();
int[] DK = DISHES.Select(d => d.Kcal).ToArray();
string[] DCAT_SET = { "breakfast", "lunch", "dinner" };
int ND = DISHES.Length;
int D = 3;                              // 3 jours (lundi..mercredi) -- borne pour un runtime notebook
int DAY_LO = 1500, DAY_HI = 1700;      // fenetre kcal/jour (3 plats)
int nBf = DC.Count(c => c == "breakfast");
int nLu = DC.Count(c => c == "lunch");
int nDi = DC.Count(c => c == "dinner");
Console.WriteLine($"Corpus : {ND} plats ({nBf} bf, {nLu} lunch, {nDi} dinner). Plan sur D={D} jours, fenetre [{DAY_LO},{DAY_HI}] kcal/jour.");


Corpus : 9 plats (3 bf, 3 lunch, 3 dinner). Plan sur D=3 jours, fenetre [1500,1700] kcal/jour.


In [8]:
// (5) Plan hebdomadaire : matrice booléenne Sel[j,i] (jour j, plat i). Z3 raisonne sur
//     TOUS les jours simultanément -- ce qu'un glouton jour-par-jour ne sait pas faire (prong B).
Solver Sw = ctx.MkSolver();
BoolExpr[,] Sel = new BoolExpr[D, ND];
for (int j = 0; j < D; j++)
    for (int i = 0; i < ND; i++)
        Sel[j, i] = ctx.MkBoolConst($"S_{j}_{i}");
// (1) 1 plat par categorie et par jour.
for (int j = 0; j < D; j++)
    foreach (var cat in DCAT_SET)
        ExactementUn(Sw, Enumerable.Range(0, ND).Where(i => DC[i] == cat).Select(i => Sel[j, i]).ToArray());
// (2) Variete globale : chaque plat au plus 1 fois dans la semaine.
for (int i = 0; i < ND; i++) {
    ArithExpr[] once = Enumerable.Range(0, D).Select(j => (ArithExpr)ctx.MkITE(Sel[j, i], ctx.MkInt(1), ctx.MkInt(0))).ToArray();
    Sw.Assert(ctx.MkLe(ctx.MkAdd(once), ctx.MkInt(1)));
}
// (3) Fenetre kcal par jour.
for (int j = 0; j < D; j++) {
    ArithExpr[] calJ = Enumerable.Range(0, ND).Select(i => (ArithExpr)ctx.MkITE(Sel[j, i], ctx.MkInt(DK[i]), ctx.MkInt(0))).ToArray();
    ArithExpr dayCal = ctx.MkAdd(calJ);
    Sw.Assert(ctx.MkAnd(ctx.MkGe(dayCal, ctx.MkInt(DAY_LO)), ctx.MkLe(dayCal, ctx.MkInt(DAY_HI))));
}
var swp = Stopwatch.StartNew();
var statusW = Sw.Check();
swp.Stop();
Console.WriteLine($"\n=== Plan hebdomadaire SMT (D={D} jours, resolu en {swp.Elapsed.TotalMilliseconds:F1} ms) ===");
if (statusW == Status.SATISFIABLE) {
    var mw = Sw.Model;
    for (int j = 0; j < D; j++) {
        var picked = Enumerable.Range(0, ND)
            .Where(i => ((BoolExpr)mw.Evaluate(Sel[j, i], true)).BoolValue == Z3_lbool.Z3_L_TRUE).ToList();
        var day = picked.Select(i => DISHES[i].Nom).ToList();
        int kcal = picked.Sum(i => DK[i]);
        Console.WriteLine($"  Jour {j + 1} : {string.Join(", ", day)}  ->  {kcal} kcal");
    }
} else {
    Console.WriteLine("  Aucun plan (UNSAT) -- relacher les contraintes.");
}



=== Plan hebdomadaire SMT (D=3 jours, resolu en 12,6 ms) ===


  Jour 1 : Porridge fruits rouges, Wrap poulet, Poulet curry  ->  1500 kcal


  Jour 2 : Yaourt granola, Quiche epinards, Risotto champignons  ->  1700 kcal


  Jour 3 : Oeufs brouilles, Salade cesar, Saumon four  ->  1600 kcal


### Interprétation 5 — pourquoi le glouton échoue là où Z3 réussit

Le **glouton** remplit le jour 1 avec le premier trio (breakfast + lunch + dinner) qui tombe
dans la fenêtre kcal, marque ces plats comme « utilisés », puis passe au jour 2. Le problème :
ce trio local peut **consommer** les seuls plats qui rendraient le jour 3 faisable. La
combinatoire (fenêtre × variété) **n'est pas compositionnelle** — optimiser jour par jour ne
garantit pas la faisabilité globale. Z3, en revanche, considère **simultanément** les D jours :
il peut « sacrifier » l'optimum local du jour 1 pour préserver la faisabilité des jours
suivants. C'est précisément la capacité distinctive d'un solveur SMT face à une heuristique
gloutonne — l'argument central (prong B) pour la programmation par contraintes.

> **Variante optimisée** : le même plan, mais **à coût minimal** et à l'échelle (corpus de 24
> plats, D=7), est consolidé dans [16e — optimisation](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb) (§6b).


## 6. Exercices


### Exercice 1 — Ajouter une contrainte de macro-nutriments (glucides + lipides)

Étendez la base nutritionnelle avec des champs `carbs` (glucides) et `fat` (lipides), puis
ajoutez au modèle booléen (§3) une contrainte « 45-65 % des kcal en glucides » (4 kcal/g).
Combien de menus valides restent ?

*Indice* : `total_carbs_g = MkAdd de MkITE(sel_i, carbs_i, 0)`, puis
`total_carbs_g * 4 >= 0.45 * total_cal` et `<= 0.65 * total_cal`. Multipliez par 100 pour
rester en arithmétique entière.


In [9]:
// Exercice 1 : contrainte macro glucides (45-65% des kcal, 4 kcal/g).
// TODO etudiant :
// Etape 1 : ajouter carbs_g a chaque aliment de FOODS.
// Etape 2 : calculer total_carbs_g via MkAdd de MkITE(sel_i, carbs_i, 0).
// Etape 3 : ajouter total_carbs_g*4 >= 0.45*total_cal ET <= 0.65*total_cal (x100 pour entiers).
// Etape 4 : compter les solutions (negation du modele + boucle).
Console.WriteLine("Exercice 1 a completer : contrainte macro glucides (45-65%).");


Exercice 1 a completer : contrainte macro glucides (45-65%).


### Exercice 2 — Plan hebdomadaire à **variété maximale**

Reconstruisez la matrice booléenne `Sel[j, i]` du §5 (9 plats, `D = 3`, variété ≤ 1), puis
**changez l'objectif** : au lieu de la faisabilité, **maximisez le nombre de plats distincts**
servis sur la semaine. Combien des 9 plats le solveur parvient-il à utiliser au moins une fois ?
Pourquoi la contrainte « 1 plat par catégorie et par jour » fixe-t-elle un plafond théorique à
cette variété ?

*Indice* : `distinct_count = MkAdd de MkITE(Or(Sel[j,i] pour tout j), 1, 0) pour tout i`,
puis `opt.MkMaximize(distinct_count)`. Trois usages du même encodage booléen : faisabilité
(§5), coût minimal (§4), variété maximale (ici). La version à l'échelle (corpus élargi, D=7)
est traitée dans [16e](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb).


In [10]:
// Exercice 2 : plan hebdomadaire a VARIETE MAXIMALE (MkMaximize).
// TODO etudiant :
// Etape 1 : reconstruire la matrice Sel[j,i] du S5 (9 plats, D=3, variete <= 1).
// Etape 2 : distinct_count = MkAdd de MkITE(Or(Sel[j,i] pour tout j), 1, 0) pour tout i.
// Etape 3 : opt.MkMaximize(distinct_count) ; afficher le plan + le nombre de plats distincts.
Console.WriteLine("Exercice 2 a completer : variete maximale sur la matrice du S5 (maximiser les plats distincts).");


Exercice 2 a completer : variete maximale sur la matrice du S5 (maximiser les plats distincts).


### Exercice 3 — Démontrer l'échec du glouton (prong B)

Implémentez le glouton décrit au §5 (remplir jour par jour avec le premier trio valide
restant) sur le corpus à 9 plats avec `D = 3`. Montrez qu'il **échoue** (se bloque avant le
jour 3) là où Z3 trouve un plan. Que se passe-t-il si vous changez l'ordre des plats dans le
corpus ?

*Indice* : boucle `for (int j = 0; j < D; j++)`, à chaque jour parcourir les plats dans
l'ordre, prendre le premier breakfast + lunch + dinner dont la somme kcal est dans
`[DAY_LO, DAY_HI]` et qui n'est pas déjà utilisé. Si aucun trio — échec. Comparer au plan SMT.


In [11]:
// Exercice 3 : glouton sequential sur D=3 jours, demontrer qu'il se bloque.
// TODO etudiant :
// Etape 1 : parcourir les jours en sequence.
// Etape 2 : pour chaque jour, premier trio (bf+lunch+dinner) valide et non-utilise.
// Etape 3 : si aucun trio -> echec (Console.WriteLine). Comparer au plan SMT de la cellule precedente.
Console.WriteLine("Exercice 3 a completer : demontrer l'echec du glouton vs SMT.");


Exercice 3 a completer : demontrer l'echec du glouton vs SMT.


## Conclusion

Ce notebook modélise le **meal-planner déclaratif** en **C# (Microsoft.Z3 natif)**, en montrant trois
encodages du même problème (index / énumération / booléen), l'optimisation du coût via
`MkMinimize`, et un **plan hebdomadaire** à matrice booléenne qui démontre la supériorité du
solveur sur une heuristique gloutonne (prong B : la combinatoire fenêtre × variété n'est pas
compositionnelle). Le jumeau Python (z3-py) du même contenu est
[Z3-16-Meal-Planner-Python](Z3-16-Meal-Planner-Python.ipynb).

**Leçons clés** :

1. **L'encodage booléen** (`MkITE(sel_i, val_i, 0)`) rend la relation sélection-caractéristique
   native, là où l'index demande une table de lookup.
2. **`MkOptimize`** sous-traite la recherche d'optimum (un appel) là où la dichotomie manuelle
   en demanderait N.
3. **Le couplage multi-jours** (variété globale + fenêtre par jour) est exactement le genre de
   problème où un solveur SMT bat une heuristique : la décision locale optimale n'est pas
   globalement faisable.

Ce notebook est le premier d'une série appliquée : les 16b-16e (données externes, capstone
patient, convergence à l'échelle, optimisation) étendent ce socle vers des données réelles et
la passage-à-l'échelle.

← [15 - Tableaux imbriqués et grilles 2D](Z3-15-Nested-Arrays-2D-Python.ipynb) | [README Z3](README.md) | [16b - couche de données >>](Z3-16b-Meal-Planner-Data-External-Python.ipynb)

***

> **Suite de la track (corpus réel).** Ce notebook reste sur un corpus **jouet**. L'idiom
> `MkPBEq` (pseudo-booléen natif) utilisé ici pour `ExactementUn` est la clé qui rend le
> meal-planner **tractable à l'échelle réelle** (~594 recettes) : cf.
> [16b](Z3-16b-Meal-Planner-Data-External-Python.ipynb) (données),
> [16c](Z3-16c-Meal-Planner-Patient-Capstone-Python.ipynb) (capstone patient),
> [16d](Z3-16d-Meal-Planner-Convergence-Scale-Python.ipynb) (convergence : l'encodage décide),
> [16e](Z3-16e-Meal-Planner-Optimize-CSharp.ipynb) (optimisation + MaxSAT + Pareto).
